# Φάση 2

## Προετοιμασία

In [1]:
#!pip -q install sentence-transformers
#!pip install faiss-cpu
#!pip install pandas
#!pip install ipywidgets

import os
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

## Φόρτωση συλλογής (documents.csv) και queries (queries.csv)

In [2]:
docs = pd.read_csv("documents.csv")   # στήλες: ID, Text
queries = pd.read_csv("queries.csv")  
print(docs)
print("Σύνολο εγγράφων:", len(docs))
print(queries)
print("Σύνολο ερωτημάτων:", len(queries))

docs["ID"] = docs["ID"].astype(str)
queries["ID"] = queries["ID"].astype(str)
docs["Text"] = docs["Text"].astype(str)
queries["Text"] = queries["Text"].astype(str)

doc_ids = docs["ID"].tolist()
doc_texts = docs["Text"].tolist()

           ID                                               Text
0      193157  Support towards the Europe PMC initiative-Cont...
1      193158  Support to the Vice-Presidents of the ERC Scie...
2      193159  Implementation of activities described in the ...
3      193160  Monitoring Atmospheric Composition and Climate...
4      193161  Pre-Operational Marine Service Continuity in T...
...       ...                                                ...
18311  215952  Better Rural Innovation: Linking Actors, Instr...
18312  215953  Pan-european assessment, monitoring, and mitig...
18313  215954  SMAll RuminanTs breeding for Efficiency and Re...
18314  215955  Networking, partnerships and tools to enhance ...
18315  215956  Towards Innovation - driven and smart solution...

[18316 rows x 2 columns]
Σύνολο εγγράφων: 18316
    ID                                               Text
0  Q01  EUTRAVEL Optimodal European Travel Ecosystem E...
1  Q02  Track And Know Big Data for Mobility Tracking .

## Μοντέλο all-MiniLM-L6-v2

In [3]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # 384 dims

##  Doc embeddings (αποθήκευση για επαναχρησιμοποίηση-cache)

In [4]:
emb_path = "doc_embeddings.npy"

if os.path.exists(emb_path):
    doc_emb = np.load(emb_path)
else:
    doc_emb = model.encode(
        doc_texts,
        batch_size=64,              # CPU-friendly
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True   # cosine
    )
    np.save(emb_path, doc_emb)

doc_emb = doc_emb.astype("float32")
print("Doc embeddings:", doc_emb.shape)

Doc embeddings: (18316, 384)


## Ευρετήριο FAISS (συνημιτονοειδής ομοιότητα μέσω inner product, IP)

In [5]:
dim = doc_emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_emb)

def write_run_file(k: int, run_path: str):
    run_name = f"minilm_faiss_cos_k{k}"
    with open(run_path, "w", encoding="utf-8") as f:
        for _, row in queries.iterrows():
            qid = row["ID"]
            qtext = row["Text"]

            q_emb = model.encode(
                [qtext],
                convert_to_numpy=True,
                normalize_embeddings=True
            ).astype("float32")

            scores, idxs = index.search(q_emb, k)  # (1,k)
            scores = scores[0]
            idxs = idxs[0]

            for rank, (doc_idx, score) in enumerate(zip(idxs, scores), start=1):
                docid = doc_ids[int(doc_idx)]
                f.write(f"{qid} Q0 {docid} {rank} {float(score)} {run_name}\n")


## Εκτελούμε την συνάρτηση για τα τρία k: 20, 30, 50

In [6]:
RUNS_DIR = "runs_phase2"
os.makedirs(RUNS_DIR, exist_ok=True)

write_run_file(20, os.path.join(RUNS_DIR, "run_k20.txt"))
write_run_file(30, os.path.join(RUNS_DIR, "run_k30.txt"))
write_run_file(50, os.path.join(RUNS_DIR, "run_k50.txt"))


print("Έτοιμα: run_k20.txt, run_k30.txt, run_k50.txt στον φάκελο ", RUNS_DIR)

Έτοιμα: run_k20.txt, run_k30.txt, run_k50.txt στον φάκελο  runs_phase2
